# EU-Park - Customer / Season-Pass EDA

**Consulting project for the theme park "EU-Park" (board: CEO + CTO).**

This notebook explores the **season-pass customer data** (`EU-Park-Customers.csv`) for goal **(c)**:

> *Predict which customers buy a season pass ("Gold" / "Silver") and understand what drives the decision.*

The snapshot reflects each customer's situation on **1 January** - the day they decided whether to buy a pass. This notebook covers the **exploratory phase**: data-quality checks, cleaning, and first insights that set up the classification model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 4.5)

In [ ]:
customers = pd.read_csv("EU-Park-Customers.csv")
print("Shape:", customers.shape, "\n")
print(customers.dtypes, "\n")
customers.head()

## 1. First look & data quality

In [ ]:
# Summary statistics for numeric + categorical columns
display(customers.describe(include="all").T)

print("Missing values per column:")
print(customers.isna().sum())
print("\nFull duplicate rows :", int(customers.duplicated().sum()))
print("Duplicate phone nums:", int(customers.Telephone_Number.duplicated().sum()))

### The target column `Pass_Type`

`Pass_Type` is missing for 4,372 rows. These are **not missing data** - they are customers who simply **did not buy a pass**. We recode the NaNs to `"None"`.

`Telephone_Number` is an ID-like field (essentially unique: 9,999 distinct values out of 10,000) and carries no predictive signal, so it is dropped.

In [ ]:
print("Pass_Type raw value counts (NaN included):")
print(customers.Pass_Type.value_counts(dropna=False))

In [ ]:
cust = customers.copy()
cust["Pass_Type"] = cust["Pass_Type"].fillna("None")
cust = cust.drop(columns=["Telephone_Number"])

# Binary target as well: did the customer buy ANY pass?
cust["AnyPass"] = (cust["Pass_Type"] != "None").astype(int)

print(cust.Pass_Type.value_counts(), "\n")
print("Class shares:")
print((cust.Pass_Type.value_counts(normalize=True) * 100).round(1).astype(str) + " %")
print("\nAny-pass rate:", f"{cust.AnyPass.mean():.1%}")
print("Remaining missing values:", int(cust.drop(columns='AnyPass').isna().sum().sum()))

## 2. Feature distributions

In [ ]:
num_cols = ["Age", "Family_Members_with_Passes", "Previously_Owned_Passes",
            "Distance_from_Park_km"]
cust[num_cols].hist(bins=30, figsize=(11, 7), color="#369")
plt.suptitle("Distributions of numeric features"); plt.tight_layout(); plt.show()

print("Club_Member share:")
print((cust.Club_Member.value_counts(normalize=True) * 100).round(1).astype(str) + " %")

## 3. What separates buyers from non-buyers?

We compare each feature across the three classes (`None`, `Silver`, `Gold`).

In [ ]:
order = ["None", "Silver", "Gold"]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.boxplot(data=cust, x="Pass_Type", y="Age", order=order, ax=axes[0,0])
axes[0,0].set_title("Age by pass type")

sns.boxplot(data=cust, x="Pass_Type", y="Distance_from_Park_km", order=order, ax=axes[0,1])
axes[0,1].set_ylim(0, 120); axes[0,1].set_title("Distance from park (km) by pass type")

(cust.groupby("Pass_Type").Previously_Owned_Passes.mean().reindex(order)
     .plot.bar(ax=axes[1,0], color="#693"))
axes[1,0].set_title("Avg previously-owned passes"); axes[1,0].tick_params(axis="x", rotation=0)

(cust.groupby("Pass_Type").Family_Members_with_Passes.mean().reindex(order)
     .plot.bar(ax=axes[1,1], color="#963"))
axes[1,1].set_title("Avg family members with passes"); axes[1,1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# Club membership vs pass type
ct = pd.crosstab(cust.Pass_Type, cust.Club_Member, normalize="index").reindex(order)
ct.plot.bar(figsize=(8, 4))
plt.title("Club-membership share by pass type"); plt.xticks(rotation=0)
plt.ylabel("share within class"); plt.legend(title="Club_Member")
plt.tight_layout(); plt.show()
display(ct.round(3))

## 4. Feature correlations with buying a pass

In [ ]:
numc = cust.assign(Club_Member=cust.Club_Member.astype(int))[
    ["AnyPass", "Age", "Family_Members_with_Passes", "Previously_Owned_Passes",
     "Club_Member", "Distance_from_Park_km"]]
plt.figure(figsize=(7, 5))
sns.heatmap(numc.corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Customer features vs AnyPass"); plt.tight_layout(); plt.show()

print("Correlation of each feature with AnyPass (sorted):")
print(numc.corr()["AnyPass"].drop("AnyPass").sort_values(ascending=False).round(3))

## 5. Non-linear effects: purchase rate by feature bin

Pearson correlation only sees *linear* relationships - that is why `Age` scored ~0.00 above.
Here we bin each feature and plot the **share of customers who buy** within each bin, which
reveals non-monotonic shapes. We look at both targets: the binary buy-rate, and the full
`None`/`Silver`/`Gold` mix.

In [ ]:
# Binary buy-rate within bins of each feature
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, col in [(axes[0,0], "Age"), (axes[0,1], "Distance_from_Park_km")]:
    b = pd.cut(cust[col], bins=6)
    cust.groupby(b, observed=True).AnyPass.mean().plot.bar(ax=ax, color="#369")
    ax.set_title(f"Buy-rate by {col}"); ax.set_ylabel("P(buy)")
    ax.tick_params(axis="x", rotation=45)

for ax, col in [(axes[1,0], "Previously_Owned_Passes"),
                (axes[1,1], "Family_Members_with_Passes")]:
    cust.groupby(col).AnyPass.mean().plot.bar(ax=ax, color="#693")
    ax.set_title(f"Buy-rate by {col}"); ax.set_ylabel("P(buy)")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# 3-class mix (None/Silver/Gold) within bins of Age and Distance
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = ["#bbbbbb", "#9fb3c8", "#d4a017"]  # None, Silver, Gold
for ax, col in [(axes[0], "Age"), (axes[1], "Distance_from_Park_km")]:
    b = pd.cut(cust[col], bins=6)
    comp = (cust.groupby(b, observed=True).Pass_Type
                .value_counts(normalize=True).unstack()[order])
    comp.plot.bar(stacked=True, ax=ax, color=colors)
    ax.set_title(f"Pass-type mix by {col}"); ax.set_ylabel("share within bin")
    ax.tick_params(axis="x", rotation=45); ax.legend(title="", fontsize=8)
plt.tight_layout(); plt.show()

## 6. Feature interactions

The decision is likely driven by *combinations* of variables. Each heatmap shows the
**buy-rate** across two binned features at once - if the colour pattern is not just
horizontal/vertical bands, the two features interact.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distance x Club_Member
dist_bin = pd.cut(cust.Distance_from_Park_km, bins=[0, 10, 20, 40, 80, 300],
                  labels=["0-10", "10-20", "20-40", "40-80", "80+"])
piv1 = (cust.assign(DistBin=dist_bin)
            .pivot_table(index="DistBin", columns="Club_Member",
                         values="AnyPass", aggfunc="mean", observed=True))
sns.heatmap(piv1, annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1, ax=axes[0])
axes[0].set_title("Buy-rate: Distance x Club_Member")

# Previously_Owned x Family_Members
piv2 = cust.pivot_table(index="Previously_Owned_Passes",
                        columns="Family_Members_with_Passes",
                        values="AnyPass", aggfunc="mean")
sns.heatmap(piv2, annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("Buy-rate: Previously_Owned x Family_Members")
plt.tight_layout(); plt.show()

## 7. Gold vs Silver - a separate question

Sections 4-6 explained *whether* someone buys. But "what drives the decision" also covers
*which tier* they choose. Here we look **only at buyers** and ask what separates `Gold`
from `Silver`.

In [ ]:
buyers = cust[cust.Pass_Type != "None"].copy()
buyers["IsGold"] = (buyers.Pass_Type == "Gold").astype(int)
print(f"Buyers: {len(buyers)} | Gold share among buyers: {buyers.IsGold.mean():.1%}")

feat = ["Age", "Family_Members_with_Passes", "Previously_Owned_Passes",
        "Distance_from_Park_km"]
print("\nMean feature value - Gold vs Silver:")
display(buyers.groupby("Pass_Type")[feat].mean().T.round(2))

bnum = buyers.assign(Club_Member=buyers.Club_Member.astype(int))[["IsGold"] + feat + ["Club_Member"]]
print("Correlation with IsGold (among buyers), sorted:")
print(bnum.corr()["IsGold"].drop("IsGold").sort_values(ascending=False).round(3))

## 8. Model-based importance

Univariate views can miss interactions. A fitted model ranks features while accounting for
them. We report three complementary measures for **both** targets:

- **Mutual information** - model-free, captures non-linear dependence.
- **Random-forest permutation importance** - drop in test accuracy when a feature is shuffled
  (the textbook answer to *"which variables enable a good prediction"*).
- **Standardised logistic-regression coefficients** (binary target) - give the *direction* of each effect.

Accuracies are shown only to gauge how much signal exists (vs. the majority-class baseline);
this is exploration, not the final tuned model.

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ["Age", "Family_Members_with_Passes", "Previously_Owned_Passes",
            "Club_Member", "Distance_from_Park_km"]
X = cust[features].assign(Club_Member=cust.Club_Member.astype(int))

# ---- Binary target: AnyPass ----
y = cust.AnyPass
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
perm = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=42, n_jobs=-1)
sc = StandardScaler().fit(Xtr)
lr = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), ytr)

report_bin = pd.DataFrame({
    "MutualInfo":        mutual_info_classif(X, y, random_state=42),
    "RF_PermImportance": perm.importances_mean,
    "LogReg_coef_std":   lr.coef_[0],
}, index=features).sort_values("RF_PermImportance", ascending=False)

print(f"BINARY (AnyPass) | RF test acc: {rf.score(Xte, yte):.3f} | "
      f"majority baseline: {max(y.mean(), 1 - y.mean()):.3f}")
display(report_bin.round(4))
report_bin["RF_PermImportance"].sort_values().plot.barh(color="#369", figsize=(7, 3))
plt.title("RF permutation importance - binary target"); plt.tight_layout(); plt.show()

In [ ]:
# ---- 3-class target: None / Silver / Gold ----
y3 = cust.Pass_Type
Xtr, Xte, ytr, yte = train_test_split(X, y3, test_size=0.25, random_state=42, stratify=y3)

rf3 = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
perm3 = permutation_importance(rf3, Xte, yte, n_repeats=10, random_state=42, n_jobs=-1)

report_3c = pd.DataFrame({
    "MutualInfo":        mutual_info_classif(X, y3, random_state=42),
    "RF_PermImportance": perm3.importances_mean,
}, index=features).sort_values("RF_PermImportance", ascending=False)

print(f"3-CLASS (None/Silver/Gold) | RF test acc: {rf3.score(Xte, yte):.3f} | "
      f"majority baseline: {y3.value_counts(normalize=True).max():.3f}")
display(report_3c.round(4))

## 9. Takeaways - what drives the season-pass decision

**Data prep:** `Pass_Type` NaN -> `"None"` (None 43.7% / Silver 28.3% / Gold 28.0%);
`Telephone_Number` dropped (ID); no remaining missing values or duplicate rows.

**The signal is strong and predictable.** A plain random forest already reaches:
- **Binary** (buy / not): **~0.91** test accuracy vs. 0.56 majority baseline.
- **3-class** (None/Silver/Gold): **~0.83** vs. 0.44 baseline.

**Variables that drive *whether* a customer buys** (consistent across correlation,
mutual information and RF permutation importance):

| Rank | Variable | Effect |
|------|----------|--------|
| 1 | `Previously_Owned_Passes` | **+** strongest driver - past owners renew |
| 2 | `Family_Members_with_Passes` | **+** households with passes buy more |
| 3 | `Distance_from_Park_km` | **-** the farther away, the less likely (logistic coef ~ -4) |
| 4 | `Club_Member` | **+** 46% of Gold buyers are club members vs 5% of non-buyers |
| 5 | `Age` | **~0 - carries essentially no information** (MI = 0.00 even non-linearly) |

**Interactions (Section 6):** club membership and short distance reinforce each other;
loyalty (`Previously_Owned`) combined with family passes pushes buy-rate highest.

**Gold vs Silver (Section 7):** among buyers (≈50% pick Gold), the tier is driven by
`Family_Members_with_Passes` (Gold 1.37 vs Silver 0.71), `Club_Member`, and
`Previously_Owned_Passes` - i.e. **loyal, family-oriented, nearby club members upgrade to Gold.**

**Key insight to flag for the board:** `Age` is *not* a useful predictor here -
the decision is about **loyalty, household, proximity and membership**, not life stage.

**Recommended next step:** build the predictive model on these features (drop or
de-prioritise `Age`); use Logistic Regression as an explainable baseline and a tuned
Random Forest / XGBoost for best accuracy, reporting permutation importance to the CEO/CTO.

> A clean modelling dataframe `cust` is ready (target recoded, ID dropped, `AnyPass` added).